

## COMM3180 Spring 2026 - _Stories from Data_

### Instructor: Matt O'Donnell (mbod@asc.upenn.edu)

------------


## Shooting victims in Philadelphia

![](img/shootings_heatmap.png)



### Exploring the dataset

* Data is available from opendataphily.org
    * https://www.opendataphilly.org/dataset/shooting-victims
    <br/><br/>


* [Metadata information](http://metadata.phila.gov/#home/datasetdetails/5719551277d6389f3005a610/representationdetails/5719551277d6389f3005a614/)


* CSV version downloaded 03/16/26

    * `data/shootings.csv`
    <br/><br/>

* Related Data 
    * Philadelphia Police Districts
        * https://www.opendataphilly.org/dataset/police-districts
        
        
---

* Related articles
    * https://www.theguardian.com/us-news/2021/sep/27/us-murder-rate-increase-2020
    * https://6abc.com/philadelphia-murder-homicides-gun-violence-shooting/11050068/

### Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import geopandas as gpd

import warnings
warnings.simplefilter('ignore')

### 1. Load the data into a data frame

In [ ]:
shoot_df = pd.read_csv('data/shootings.csv') 

### 2. Investigate the size of the data and _eyeball_ some rows
* Key questions
    1. What does each row in the dataset represent?
    2. Which columns central to exploring stories in these data?
    3. What kinds of questions are possible given the data available (e.g. if age/race/ethnicity/other demographic information is available)?

In [ ]:
shoot_df.shape

In [ ]:
shoot_df.columns

* Some of these columns are not mention in the metadata entry that lists 20 columns

* Looks like there could be duplication (e.g. `point_x` and `lng`, `point_y` and `lat` etc) and extra fields added for a specific dashboard/application use.

In [ ]:
shoot_df.head()

* looks like each row is a shooting incident

* looks like geo location duplicate across four columns
    * `lng` = `point_x`
    * `lat` = `point_y`
    
* question: do more shootings occur inside or outside?
* who do demographic columns refer to:
    * e.g. age, sex, race - offender or victim?

* What are `the_geom` and `the_geom_webmercator` columns?


* Do we need them?

In [ ]:
shoot_df['the_geom_webmercator']

In [ ]:
shoot_df['the_geom'].head(10)

* Looks like some encoded information for a specific application (probably related to geodata)


* We will ignore for now

### 3. Look at columns and kinds and range of data captured in dataset

In [ ]:
shoot_df.columns

* what is `dist`? - distance? from what? - probably district
    * checking the metadata confirms that it is __POLICE DISTRICT__

In [ ]:
# how many values in the dist column?
shoot_df['dist'].nunique()

In [ ]:
# what are the distinct values?
shoot_df['dist'].unique()

In [ ]:
# how are they distributed?
shoot_df['dist'].value_counts()

* We will come back and look at the file `data/Boundaries_District.csv` in more detail. But if we load it and take a quick look:

In [ ]:
districts_df = pd.read_csv('data/Boundaries_District.csv')

In [ ]:
districts_df.shape

In [ ]:
districts_df.head()

In [ ]:
districts_df['DIST_NUM'].nunique()

In [ ]:
districts_df['DIST_NUM'].unique()

* Seems like the `dist` values in the `shoot_df` and those in the `districts_df` align (although one more?)

#### Returning to columns in `shoot_df`

* The `race` column

In [ ]:
shoot_df['race'].value_counts()

* Values:
    * B = black
    * W = white
    * A = Asian(?)
    * U=?unknown/unidentified
    * I = indigenous?


* Checking the entry from the metadata field table


| Field Name | Alias | Description | Type |
| --- | --- | --- | --- |
| RACE | Race| The race of the individual shot (Asian, Black, White, American Indian, Other) |	Text |


* __NOTE: We should map these single letter abbreviations to readable labels__


* Probably doesn't help us understand the distribution beyond what we see from using `.value_counts()` but we could do some quick visualization

In [ ]:
shoot_df['race'].value_counts().plot(kind='bar')
plt.show()

#### WARNING DON'T DO THIS!!!!

* For illustration purposes of how horrible a pie chart is for understanding data take a look at this

In [ ]:
shoot_df['race'].value_counts().plot(kind='pie')
plt.show()

* `pandas` will let you create a pie-chart... but it is strongly discouraged as most data visualization sources will tell you how poorly they communicate proportions. Best to use a bar chart. Enough said!


* The `year` column

In [ ]:
shoot_df['year'].value_counts()


* The `date_` column

 
* Looking at a random sample of 20 rows:

In [ ]:
shoot_df['date_'].sample(20)

* `date_` has a `YYYY-MM-DD` format


* The `age` column


* Looks numeric for years old, so can use `.describe() to look at min, max, mean etc.

In [ ]:
shoot_df['age'].describe()

* We could also use `.value_counts()` as it is a fixed set (0-117 years old)

In [ ]:
shoot_df['age'].value_counts()

* And perhaps use a plot of this distribution

In [ ]:
shoot_df['age'].value_counts().plot(kind='bar', figsize=(16,4))
plt.show()

* 20-25 year olds most frequent 


* But this is not that helpful to observe patterns. 


* Below we will use a histogram instead

### 4. Identify the dimensions that could be used for grouping and summarizing

* Look at the types of values in the columns:
    * categorical - groups, kinds, outcomes, etc.
        * `sex`
        * `race` - ?latino col is separate
        * type of shooting
            * `fatal`
            * `location` - inside or not
        * officer involved
        * wound type
    * date
        * `year`
        * `date_` y-m-d
    * location
        * geolocation columns (`point_x`, `lng` etc)
        * `dist` - polic district
    * continuous counts, measurements
        * `age` - but need to add grouping

### 5. Summarize key columns

* Use Pandas functions like `.value_counts()` and `.describe()` and `.count()` to look at the distribution of values


* For continuous values you can use `.plot()` and `.hist()`


* Also try `.plot(kind=bar)` on the result of counting values.

In [ ]:
shoot_df['age'].plot(kind='hist', bins=50)
plt.show()

In [ ]:
shoot_df['age'].plot(kind='density')
plt.show()

In [ ]:
shoot_df['age'].mean()

In [ ]:
shoot_df['age'].median()

In [ ]:
shoot_df['age'].describe()

In [ ]:
shoot_df['race'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
shoot_df['inside'].value_counts()

In [ ]:
shoot_df['outside'].value_counts()

In [ ]:
race_mapping = {
    'B': 'BLACK',
    'W': 'WHITE',
    'A': 'ASIAN',
    'I': 'INDIGENOUS',
    'U': 'UNIDENTIFIED/OTHER'
}

### Remapping values with `.map()` function

In [ ]:
shoot_df['race'].map(race_mapping).head(10)

In [ ]:
race_mapping['B']

In [ ]:
shoot_df['race'].head(10)

In [ ]:
shoot_df['race_long']=shoot_df['race'].map(race_mapping)

In [ ]:
shoot_df.shape

In [ ]:
shoot_df[['race','race_long']]

In [ ]:
shoot_df['latino'].value_counts()

In [ ]:
latino_mapping = {
    0 : 'NON-LATINO',
    1 : 'LATINO'
}

In [ ]:
shoot_df['latino_long']=shoot_df['latino'].map(latino_mapping)

In [ ]:
shoot_df.shape

In [ ]:
shoot_df['ethnicity']=shoot_df['race_long'] + ' ' + shoot_df['latino_long']

In [ ]:
shoot_df['ethnicity'].head(20)

In [ ]:
shoot_df['ethnicity'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
shoot_df.groupby(['race_long', 'latino_long']).size().unstack().plot(kind='bar', stacked=False)
plt.show()

In [ ]:
shoot_df.groupby(['race_long', 'latino_long']).size()

In [ ]:
shoot_df.count()

* not much missing data - come back and check for how missing values are recorded

In [ ]:
shoot_df.groupby('sex')['race_long'].value_counts()

In [ ]:
shoot_df[['inside','outside']].corr()

In [ ]:
shoot_df.groupby('year')['fatal'].value_counts(normalize=True)

In [ ]:
bins=[0,9,19,29,39,49,59,69,79,100]
labels=["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69","70-79", "80+"]

In [ ]:
shoot_df['age_group']=pd.cut(shoot_df['age'], bins=bins, labels=labels)

In [ ]:
shoot_df[['age', 'age_group']]

In [ ]:
shoot_df['age_group'].value_counts()

In [ ]:
shoot_df[shoot_df['age_group']=='80+']

In [ ]:
shoot_df['age'].hist(grid=False)
plt.show()

In [ ]:
shoot_df['wound'].unique()

In [ ]:
shoot_df[shoot_df['fatal']==1].groupby(['dist','wound']).size()

In [ ]:
shoot_df['code'].value_counts()

In [ ]:
# Looking at the metadata and UCS codes for classifying crimes

print(
'''
0000: Additional Victim
0100 – 0119: Homicide
0200 – 0299: Rape
0300 – 0399: Robbery
0400 – 0499: Aggravated Assault
3000 – 3900: Hospital Cases
''')

# could create a mapping to go from number to label

In [ ]:
def code2type(code):
    
    if code<120:
        incident='Homicide'
    elif code<300:
        incident='Rape'
    elif code<400:
        incident='Robbery'
    elif code<500:
        incident='Aggravated Assault'
    elif code>=3000 and code<4000:
        incident='Hospital Cases'
    else:
        incident=None
    
    return incident

In [ ]:
shoot_df['code'].map(code2type)

In [ ]:
shoot_df['code_long'] = shoot_df['code'].map(code2type)

In [ ]:
shoot_df['fatal_long']=shoot_df['fatal'].map({0: 'NON-FATAL', 1: 'FATAL'})
shoot_df['outside_long']=shoot_df['outside'].map({0: 'INSIDE', 1: 'OUTSIDE'})

In [ ]:
shoot_df['date_YM']=shoot_df['date_'].str[:7]

In [ ]:
shoot_df.groupby('date_YM').size().plot(figsize=(16,4))
plt.show()

In [ ]:
shoot_YM_race = shoot_df.groupby(['date_YM','race_long']).size().unstack().fillna(0)

shoot_YM_race.plot(kind='line',figsize=(16,4), stacked=False)
plt.show()

In [ ]:
shoot_df.groupby('date_YM').size().plot(figsize=(16,4), kind='bar')
plt.show()

In [ ]:
shoot_YM_fatal=shoot_df.groupby(['date_YM','fatal_long']).size().unstack()
shoot_YM_fatal.plot(figsize=(16,4), kind='bar', stacked=True)
plt.show()

### Looking at Polic Districts

In [ ]:
shoot_by_dist=shoot_df['dist'].value_counts()

In [ ]:
ppd_dist_gdf = gpd.read_file('data/Boundaries_District.geojson')

In [ ]:
ppd_dist_gdf.head()

In [ ]:
ppd_dist_gdf['centX']=ppd_dist_gdf.centroid.x
ppd_dist_gdf['centY']=ppd_dist_gdf.centroid.y

In [ ]:
ppd_dist_gdf.plot(color='white', edgecolor='gray', figsize=(10,10))

ppd_dist_gdf.apply(lambda r: plt.text(r['centX'], r['centY'], r['DIST_NUM']),axis=1)
plt.show()

In [ ]:
ppd_dist_counts_gdf = ppd_dist_gdf.merge(shoot_by_dist, left_on='DIST_NUM', right_index=True) 

In [ ]:
ppd_dist_counts_gdf

In [ ]:
ppd_dist_counts_gdf.plot(edgecolor='gray', column='count', figsize=(10,10), 
                         legend=True, cmap='Oranges')

ppd_dist_counts_gdf.apply(lambda r: plt.text(r['centX'], r['centY'], r['DIST_NUM']),axis=1)

plt.show()